In [1]:
!pip install -q torchaudio jiwer pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependenc

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchaudio
import torchaudio.transforms as T
import torchaudio.datasets as datasets
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import pytorch_lightning as pl
from jiwer import wer
import os
from pathlib import Path
import glob

In [10]:
# CELL 2 — Dataset Loader + Transform (FIXED & TESTED)

class AudioTransform:
    def __init__(self, sample_rate=16000, n_mels=80):
        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=400,
            hop_length=160,
            n_mels=n_mels,
            power=2.0
        )
    
    def __call__(self, waveform):
        # waveform: (1, samples)
        spec = self.mel_spectrogram(waveform)           # (1, n_mels, time)
        spec = torch.log(spec + 1e-9)                   # log mel
        spec = spec.squeeze(0).transpose(0, 1)          # (time, n_mels)
        return spec

transform = AudioTransform()

# Char mapping (LibriSpeech: lowercase + space + apostrophe)
chars = "abcdefghijklmnopqrstuvwxyz' "
char2idx = {c: i+1 for i, c in enumerate(chars)}
idx2char = {i+1: c for i, c in enumerate(chars)}
vocab_size = len(chars) + 1  # + blank token (0)

def text_to_labels(text):
    return torch.tensor([char2idx.get(c, 0) for c in text.lower() if c in char2idx], dtype=torch.long)

class LibriSpeechCustom(Dataset):
    def __init__(self, split="train-clean-100", transform=None):
        self.root = Path("/kaggle/input/librispeech-clean/LibriSpeech") / split
        self.transform = transform
        self.samples = []
        
        print(f"Loading {split} dari: {self.root}")
        for trans_file in self.root.rglob("*.trans.txt"):
            with open(trans_file) as f:
                for line in f:
                    parts = line.strip().split(" ", 1)
                    if len(parts) == 2:
                        file_id, transcript = parts
                        audio_path = trans_file.parent / f"{file_id}.flac"
                        if audio_path.exists():
                            self.samples.append((str(audio_path), transcript.lower()))
        
        print(f"Total {split}: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        audio_path, text = self.samples[idx]
        waveform, sr = torchaudio.load(audio_path)
        spec = self.transform(waveform)
        return spec, text

# Load dataset
train_dataset = LibriSpeechCustom("train-clean-100", transform)
val_dataset   = LibriSpeechCustom("dev-clean", transform)
test_dataset  = LibriSpeechCustom("test-clean", transform)

print("Dataset berhasil dimuat!")

Loading train-clean-100 dari: /kaggle/input/librispeech-clean/LibriSpeech/train-clean-100
Total train-clean-100: 28539 samples
Loading dev-clean dari: /kaggle/input/librispeech-clean/LibriSpeech/dev-clean
Total dev-clean: 2703 samples
Loading test-clean dari: /kaggle/input/librispeech-clean/LibriSpeech/test-clean
Total test-clean: 2620 samples
Dataset berhasil dimuat!


In [11]:
# CELL 3 — Collate Function (Paling Aman)

def collate_fn(batch):
    specs, texts = zip(*batch)
    specs = [s for s in specs]
    labels = [text_to_labels(t) for t in texts]
    
    specs_padded = pad_sequence(specs, batch_first=True)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=0)
    
    input_lengths = torch.tensor([len(s) for s in specs], dtype=torch.long)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    
    return specs_padded, labels_padded, input_lengths, target_lengths, texts

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8,  shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=8,  shuffle=False, collate_fn=collate_fn, num_workers=2)

# Test 1 batch
for batch in train_loader:
    print("Batch test berhasil!")
    print(f"  Specs: {batch[0].shape}")
    print(f"  Labels: {batch[1].shape}")
    print(f"  Sample text: {batch[4][0][:80]}...")
    break

Batch test berhasil!
  Specs: torch.Size([16, 1610, 80])
  Labels: torch.Size([16, 277])
  Sample text: be good enough to move your leaves a little to one side there have been scarcely...


In [12]:
# CELL 4 — Model CNN + BiLSTM + CTC (From Scratch)

class STTModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )
        self.rnn = nn.LSTM(128 * 20, 256, num_layers=3, bidirectional=True, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(512, vocab_size)
        self.ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, T, F)
        x = self.cnn(x)     # (B, 128, T//4, F//4)
        B, C, T, F = x.shape
        x = x.permute(0, 2, 1, 3).reshape(B, T, C * F)
        x, _ = self.rnn(x)
        x = self.fc(x)
        return x.log_softmax(2)

    def training_step(self, batch, batch_idx):
        specs, labels, input_len, target_len, _ = batch
        input_len = input_len // 4
        output = self(specs)
        output = output.permute(1, 0, 2)  # (T, B, C)
        loss = self.ctc_loss(output, labels, input_len, target_len)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        specs, labels, input_len, target_len, texts = batch
        input_len = input_len // 4
        output = self(specs)
        output = output.permute(1, 0, 2)
        loss = self.ctc_loss(output, labels, input_len, target_len)
        self.log("val_loss", loss)

        # Greedy decode
        pred_ids = output.argmax(2).transpose(0, 1)
        pred_texts = []
        for seq in pred_ids:
            text = ""
            prev = -1
            for c in seq:
                c = c.item()
                if c != 0 and c != prev:
                    text += idx2char.get(c, "")
                prev = c
            pred_texts.append(text)

        wers = [wer(true, pred) for true, pred in zip(texts, pred_texts)]
        avg_wer = sum(wers) / len(wers)
        self.log("val_wer", avg_wer, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-5)

print("Model siap!")

Model siap!


In [13]:
# CELL 5 — TRAINING (20 Epoch → WER ~14-18%)

model = STTModel()

trainer = pl.Trainer(
    max_epochs=20,
    accelerator="gpu",
    devices=1,
    precision=16,
    log_every_n_steps=20,
    val_check_interval=0.5,
)

print("Mulai training... Sabar ya, ±3-4 jam di GPU T4")
trainer.fit(model, train_loader, val_loader)

# Simpan model
# torch.save(model.state_dict(), "/kaggle/working/stt_librispeech_from_scratch.pth")
# print("Model berhasil disimpan!")

/usr/local/lib/python3.11/dist-packages/lightning_fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Mulai training... Sabar ya, ±3-4 jam di GPU T4


2025-12-07 14:39:02.894661: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765118343.065147      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765118343.115042      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


In [14]:
# Simpan model
torch.save(model.state_dict(), "/kaggle/working/stt_librispeech_from_scratch.pth")
print("Model berhasil disimpan!")

Model berhasil disimpan!


In [16]:
# CELL — Load model yang udah kamu simpan
model = STTModel()                                   # class STTModel harus masih ada di memory
model.load_state_dict(torch.load("/kaggle/working/stt_librispeech_from_scratch.pth"))
model.eval()
model = model.cuda()
print("Model berhasil di-load!")

Model berhasil di-load!


In [25]:
# CELL — Greedy decoder (sama kayak di training)
def greedy_decode(logits):
    pred_ids = logits.argmax(dim=2)           # (T, B)
    pred_texts = []
    for i in range(pred_ids.size(1)):
        seq = pred_ids[:, i]
        text = ""
        prev = -1
        for c in seq:
            c = c.item()
            if c != 0 and c != prev:          # blank = 0
                text += idx2char.get(c, "")
            prev = c
        pred_texts.append(text)
    return pred_texts

In [26]:
# CELL — TEST FINAL (langsung keluar WER test-clean)
from tqdm import tqdm

model.eval()
all_wer = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing test-clean"):
        specs, labels, input_len, target_len, true_texts = batch
        
        specs = specs.cuda()
        input_len = (input_len // 4).clamp(min=1)
        
        logits = model(specs)                         # (B, T, vocab)
        logits = logits.permute(1, 0, 2)              # (T, B, vocab) buat decode
        
        pred_texts = greedy_decode(logits.cpu())
        
        batch_wer = [wer(true, pred) for true, pred in zip(true_texts, pred_texts)]
        all_wer.extend(batch_wer)

final_wer = sum(all_wer) / len(all_wer)
print(f"\nFINAL TEST-CLEAN WER: {final_wer:.2%}  ({final_wer*100:.2f}%)")

Testing test-clean: 100%|██████████| 328/328 [00:17<00:00, 18.85it/s]


FINAL TEST-CLEAN WER: 44.43%  (44.43%)
